# Text Tokenization: Character To Subword

A neural network cannot consume raw strings directly. We need a reversible map from text to integer IDs, and we want that map to reveal the tradeoff between transparency and compression.


## Problem: What Must This Component Compute?

At this point in the model, the input is ordinary text, but the downstream modules expect integer sequences with shape `(T,)` or `(B, T)`. The tokenization step must convert text into discrete symbols, preserve enough structure for learning, and support decoding back to readable strings.

We will first write the operation directly with transparent character IDs so every symbol boundary is visible. Only after the numerical checks pass will we bridge to subword tokenization.

Why this matters later: when an architecture paper claims to improve attention, memory, quantization, or world modeling, it is usually changing one of these constraints: what information can move across positions, how much memory is required, what objective is optimized, or which representations are preserved.


## Character Vocabularies And Notation

Let a corpus be a string `s = c_1 c_2 \ldots c_T`. A character tokenizer constructs a vocabulary `V_char = sorted(set(s))`, then builds two maps:

- `stoi`: symbol to integer
- `itos`: integer to symbol

Encoding replaces each character with its ID. Decoding performs the inverse lookup. This is educational because every token boundary is obvious, but the sequence length remains large.


In [ ]:
from pathlib import Path
from llm_from_scratch.tokenization import CharTokenizer, train_bpe_tokenizer

text = Path("../data/tiny_corpus.txt").read_text(encoding="utf-8")
char_tokenizer = CharTokenizer.from_text(text)
char_ids = char_tokenizer.encode(text[:80])
bpe_tokenizer = train_bpe_tokenizer([text], vocab_size=64)
bpe_ids = bpe_tokenizer.encode(text[:80]).ids
char_tokenizer.vocab_size, len(bpe_tokenizer.get_vocab()), char_ids[:20], bpe_ids[:20]


In [ ]:
roundtrip_text = char_tokenizer.decode(char_ids)

assert roundtrip_text == text[:80]
assert char_tokenizer.vocab_size == len(set(text))
assert len(bpe_ids) <= len(char_ids)

{"char_vocab_size": char_tokenizer.vocab_size, "sample_length": len(char_ids)}


## Why Characters Are Useful But Unrealistic

Character tokenization makes the mapping from text to IDs completely explicit, which is ideal for learning. The problem is efficiency: common words are split into many tiny steps, long contexts become longer still, and the model cannot reuse whole morphemes or word pieces as stable units.


In [ ]:
from collections import Counter

words = ["low", "lower", "lowest", "newer", "wider"]
split_words = [list(word) + ["</w>"] for word in words]
pair_counts = Counter()
for symbols in split_words:
    pair_counts.update(zip(symbols, symbols[1:]))

top_pair, top_count = pair_counts.most_common(1)[0]

def merge_once(symbols, pair):
    merged = []
    i = 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:
            merged.append(symbols[i] + symbols[i + 1])
            i += 2
        else:
            merged.append(symbols[i])
            i += 1
    return merged

merged_example = merge_once(split_words[0], top_pair)
{"top_pair": top_pair, "count": top_count, "before": split_words[0], "after": merged_example}


## BPE Intuition And Hugging Face Tokenizers

Byte-pair encoding starts from small symbols, counts adjacent pairs, and repeatedly merges the most frequent pair into a reusable subword token. The Hugging Face `tokenizers` stack automates exactly that loop with optimized implementations, pre-tokenizers, and byte-level handling.

The key conceptual change from characters is that the vocabulary grows so the sequence length can shrink.


In [ ]:
sample = text[:80]
sample_char_ids = char_tokenizer.encode(sample)
sample_bpe = bpe_tokenizer.encode(sample)
merged_tokens = sorted(token for token in bpe_tokenizer.get_vocab() if len(token) > 1)[:15]

assert char_tokenizer.decode(sample_char_ids) == sample
assert len(sample_bpe.ids) <= len(sample_char_ids)

{
    "char_token_count": len(sample_char_ids),
    "bpe_token_count": len(sample_bpe.ids),
    "example_subwords": merged_tokens,
}


## Exercise Checkpoint 1

1. Why does a character tokenizer make debugging easier than a subword tokenizer?
2. If two different strings map to the same encoded IDs, what core tokenizer property has failed?


## Exercise Checkpoint 2

1. In BPE, what information does the merge counter use to decide which pair to combine next?
2. Explain why increasing vocabulary size can reduce sequence length while also increasing the size of the embedding table.
